# Lesson 4 — ReAct & Agent Loops

## Build Your First AI Agent

In this lesson, we will build a simple Research Assistant Agent.

The agent will follow the ReAct pattern:

**Reason → Act → Observe → Repeat**

The agent will:
- Understand the user's task
- Decide what action to take
- Select a tool
- Execute the tool
- Observe the result
- Decide whether more information is needed
- Repeat when necessary
- Produce a final answer

## Learning Objectives

By the end of this lab, you will be able to:

1. Understand the ReAct agent pattern.
2. Explain the difference between reasoning, action, and observation.
3. Build a basic agent loop.
4. Let an agent select and use tools.
5. Repeat tool calls when more information is needed.
6. Add a termination condition.
7. Build a simple Research Assistant Agent.
8. Complete a multi-step task using tools.

## What is ReAct?

ReAct stands for:

**Reason + Act**

An agent does not simply answer a question immediately.

Instead, it follows a loop:

1. **Reason** — Decide what needs to be done.
2. **Act** — Use a tool or perform an action.
3. **Observe** — Look at the result.
4. **Repeat** — Decide what to do next.
5. **Terminate** — Stop when the task is complete.

### Agent Loop

User Task
↓
Reason
↓
Act
↓
Observe
↓
Reason Again
↓
Act Again
↓
Observe
↓
Final Answer

## Example: Research Assistant

Suppose the user asks:

"Research LangGraph and explain its main features and how it differs from LangChain."

The agent may perform multiple steps:

Step 1:
Search for information about LangGraph.

Step 2:
Observe the search results.

Step 3:
Search for LangGraph features.

Step 4:
Observe the results.

Step 5:
Search for LangGraph vs LangChain.

Step 6:
Observe the results.

Step 7:
Decide that enough information has been collected.

Step 8:
Generate the final answer.

This is an agent loop.

## Install Libraries

In [ ]:
!pip -q install google-genai ddgs

# Import Libraries

In [ ]:
import os
import ast
import operator as op

from google import genai
from google.genai import types
from ddgs import DDGS

print("Libraries imported successfully.")

# Add Gemini API Key

In [ ]:
import getpass

api_key = getpass.getpass("Enter your Gemini API key: ")

os.environ["GEMINI_API_KEY"] = api_key

print("API key loaded successfully.")

# Create Gemini Client

In [ ]:
client = genai.Client(
    api_key=os.environ["GEMINI_API_KEY"]
)

print("Gemini client created.")

# Select Model

In [ ]:
MODEL = "gemini-3.8-flash"

print("Model:", MODEL)

# Build the Tools

Create Web Search Tool

In [ ]:
def web_search(query: str):
    try:
        results = DDGS().text(
            query,
            max_results=3
        )

        simplified = []

        for item in results:
            simplified.append({
                "title": item.get("title"),
                "url": item.get("href"),
                "snippet": item.get("body")
            })

        return simplified

    except Exception as e:
        return {
            "error": f"Search failed: {e}"
        }

# Test Web Search

In [ ]:
results = web_search("LangGraph")

print(results)

# Tool Schema

Define Search Tool Schema

In [ ]:
search_declaration = {
    "name": "web_search",
    "description": "Searches the web for information.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "The search query."
            }
        },
        "required": ["query"]
    }
}

print("Search tool schema created.")

# Create Available Tools

In [ ]:
AVAILABLE_TOOLS = {
    "web_search": web_search
}

TOOL_DECLARATIONS = [
    search_declaration
]

print("Available tools:", list(AVAILABLE_TOOLS.keys()))

# Understanding the Agent Loop

Basic Loop Structure

## The Basic Agent Loop

Our agent will repeatedly perform these steps:

1. Send the task to Gemini.
2. Gemini decides whether a tool is needed.
3. If a tool is selected, execute it.
4. Send the tool result back to Gemini.
5. Gemini observes the result.
6. Gemini decides what to do next.
7. Repeat until a final answer is produced.

We will also use a maximum number of steps to prevent infinite loops.

# Maximum Steps

In [ ]:
MAX_STEPS = 2

print("Maximum agent steps:", MAX_STEPS)

# Build the Agent

Start Agent Function

In [ ]:
def run_react_agent(user_prompt):

    contents = [
        types.Content(
            role="user",
            parts=[
                types.Part.from_text(
                    text=user_prompt
                )
            ]
        )
    ]

    print("Agent started.")

# Add the Agent Loop

In [ ]:
def run_react_agent(user_prompt):

    contents = [
        types.Content(
            role="user",
            parts=[
                types.Part.from_text(
                    text=user_prompt
                )
            ]
        )
    ]

    config = types.GenerateContentConfig(
        tools=[
            types.Tool(
                function_declarations=TOOL_DECLARATIONS
            )
        ]
    )

    for step in range(MAX_STEPS):

        print(f"\n--- Agent Step {step + 1} ---")

        response = client.models.generate_content(
            model=MODEL,
            contents=contents,
            config=config
        )

        candidate = response.candidates[0]

        contents.append(candidate.content)

        function_calls = [
            part.function_call
            for part in candidate.content.parts
            if part.function_call
        ]

        # No tool call means the agent has produced a final answer
        if not function_calls:

            print("Agent decided that the task is complete.")

            return response.text

        tool_response_parts = []

        for function_call in function_calls:

            tool_name = function_call.name
            tool_args = dict(function_call.args)

            print("Reason → Act")
            print("Tool selected:", tool_name)
            print("Arguments:", tool_args)

            tool_function = AVAILABLE_TOOLS[tool_name]

            tool_result = tool_function(**tool_args)

            print("Observe")
            print("Tool result:", tool_result)

            tool_response_parts.append(
                types.Part.from_function_response(
                    name=tool_name,
                    response={
                        "result": tool_result
                    }
                )
            )

        contents.append(
            types.Content(
                role="user",
                parts=tool_response_parts
            )
        )

    return "The agent stopped because it reached the maximum number of steps."

# Understand What the Code Does

Agent Loop Explanation

## How the Agent Loop Works

The loop follows:

### Reason

Gemini decides what action is needed.

### Act

The selected tool is executed.

### Observe

The agent receives the tool result.

### Repeat

The agent receives the result and decides what to do next.

### Terminate

The agent stops when:

- Gemini produces a final answer, OR
- The maximum number of steps is reached.

This prevents the agent from running forever.

# First Test

Simple Research Question

Test Gemini + your search tool schema

In [ ]:
response = client.models.generate_content(
    model=MODEL,
    contents="What is LangGraph?",
    config=types.GenerateContentConfig(
        tools=[
            types.Tool(
                function_declarations=TOOL_DECLARATIONS
            )
        ]
    )
)

print(response.text)

Force a tool-use scenario

In [ ]:
response = client.models.generate_content(
    model=MODEL,
    contents="Search the web for the latest information about LangGraph.",
    config=types.GenerateContentConfig(
        tools=[
            types.Tool(
                function_declarations=TOOL_DECLARATIONS
            )
        ]
    )
)

for part in response.candidates[0].content.parts:
    if part.function_call:
        print("Tool selected:", part.function_call.name)
        print("Arguments:", dict(part.function_call.args))
    elif part.text:
        print("Text:", part.text)

Execute the selected tool

In [ ]:
tool_name = None
tool_args = None

for part in response.candidates[0].content.parts:
    if part.function_call:
        tool_name = part.function_call.name
        tool_args = dict(part.function_call.args)
        break

print("Tool:", tool_name)
print("Arguments:", tool_args)

In [ ]:
tool_result = AVAILABLE_TOOLS[tool_name](**tool_args)

print("Tool result:")
print(tool_result)

Send Observation Back to Gemini

In [ ]:
tool_response_part = types.Part.from_function_response(
    name=tool_name,
    response={
        "result": tool_result
    }
)

contents = [
    types.Content(
        role="user",
        parts=[
            types.Part.from_text(
                text="Search the web for the latest information about LangGraph."
            )
        ]
    ),
    response.candidates[0].content,
    types.Content(
        role="user",
        parts=[
            tool_response_part
        ]
    )
]

print("Observation sent back to Gemini.")

# Gemini Reasons Again

In [ ]:
final_response = client.models.generate_content(
    model=MODEL,
    contents=contents,
    config=types.GenerateContentConfig(
        tools=[
            types.Tool(
                function_declarations=TOOL_DECLARATIONS
            )
        ]
    )
)

print("Final Answer:")
print(final_response.text)

# Let's inspect what Gemini requested

In [ ]:
for part in final_response.candidates[0].content.parts:

    if part.function_call:
        print("Gemini wants another tool:")
        print("Tool:", part.function_call.name)
        print("Arguments:", dict(part.function_call.args))

    elif part.text:
        print("Gemini text:")
        print(part.text)

# Build the Automatic ReAct Loop

In [ ]:
def run_react_agent(user_prompt):

    contents = [
        types.Content(
            role="user",
            parts=[
                types.Part.from_text(
                    text=user_prompt
                )
            ]
        )
    ]

    config = types.GenerateContentConfig(
        tools=[
            types.Tool(
                function_declarations=TOOL_DECLARATIONS
            )
        ]
    )

    for step in range(MAX_STEPS):

        print(f"\n--- Agent Step {step + 1} ---")

        # REASON
        response = client.models.generate_content(
            model=MODEL,
            contents=contents,
            config=config
        )

        candidate = response.candidates[0]

        # Store Gemini's response
        contents.append(candidate.content)

        # Check whether Gemini wants to use a tool
        function_calls = [
            part.function_call
            for part in candidate.content.parts
            if part.function_call
        ]

        # TERMINATE if no tool is requested
        if not function_calls:

            print("Agent finished.")

            for part in candidate.content.parts:
                if part.text:
                    return part.text

            return "The agent finished without a text response."

        # ACT + OBSERVE
        tool_response_parts = []

        for function_call in function_calls:

            tool_name = function_call.name
            tool_args = dict(function_call.args)

            print("Reason → Act")
            print("Tool selected:", tool_name)
            print("Arguments:", tool_args)

            # Execute tool
            tool_function = AVAILABLE_TOOLS[tool_name]

            tool_result = tool_function(**tool_args)

            print("Observe")
            print("Tool result:", tool_result)

            # Prepare observation for Gemini
            tool_response_parts.append(
                types.Part.from_function_response(
                    name=tool_name,
                    response={
                        "result": tool_result
                    }
                )
            )

        # Send observations back to Gemini
        contents.append(
            types.Content(
                role="user",
                parts=tool_response_parts
            )
        )

    print("Maximum steps reached.")

    return "The agent stopped because it reached the maximum number of steps."

# Test the Agent

In [ ]:
print("Current MAX_STEPS:", MAX_STEPS)

In [ ]:
response = client.models.generate_content(
    model=MODEL,
    contents="Say hello in one short sentence."
)

print(response.text)

In [ ]:
import time

def generate_with_retry(contents, config=None, max_retries=3):

    for attempt in range(max_retries):

        try:
            return client.models.generate_content(
                model=MODEL,
                contents=contents,
                config=config
            )

        except Exception as e:

            if "503" in str(e):

                print(f"Gemini temporarily unavailable. Retrying... ({attempt + 1}/{max_retries})")
                time.sleep(5)

            else:
                raise e

    raise Exception("Gemini remained unavailable after multiple retries.")

In [ ]:
MAX_STEPS = 2

def run_react_agent(user_prompt):

    contents = [
        types.Content(
            role="user",
            parts=[
                types.Part.from_text(
                    text=user_prompt
                )
            ]
        )
    ]

    config = types.GenerateContentConfig(
        tools=[
            types.Tool(
                function_declarations=TOOL_DECLARATIONS
            )
        ]
    )

    for step in range(MAX_STEPS):

        print(f"\n--- Agent Step {step + 1} ---")

        # REASON
        response = generate_with_retry(
            contents=contents,
            config=config
        )

        candidate = response.candidates[0]

        function_calls = [
            part.function_call
            for part in candidate.content.parts
            if part.function_call
        ]

        # If Gemini gives a final text answer
        if not function_calls:

            print("Agent finished.")

            for part in candidate.content.parts:
                if part.text:
                    return part.text

            return "The agent finished without a text response."

        # Store Gemini's response
        contents.append(candidate.content)

        # ACT + OBSERVE
        tool_response_parts = []

        for function_call in function_calls:

            tool_name = function_call.name
            tool_args = dict(function_call.args)

            print("Reason → Act")
            print("Tool selected:", tool_name)
            print("Arguments:", tool_args)

            # Execute tool
            tool_function = AVAILABLE_TOOLS[tool_name]

            tool_result = tool_function(**tool_args)

            print("Observe")
            print("Tool result:", tool_result)

            # Prepare observation
            tool_response_parts.append(
                types.Part.from_function_response(
                    name=tool_name,
                    response={
                        "result": tool_result
                    }
                )
            )

        # Send observation back to Gemini
        contents.append(
            types.Content(
                role="user",
                parts=tool_response_parts
            )
        )

    # If maximum steps are reached
    print("\nMaximum tool steps reached.")

    return (
        "The agent completed the allowed research steps. "
        "Please review the search results above."
    )

In [ ]:
answer = run_react_agent(
    "Search the web for information about LangGraph and summarize your findings."
)

print("\nFinal Answer:")
print(answer)

# Try a Different Task

## Try a Different Research Task

The same agent can handle different research questions.

Let's ask it to research another AI topic.

In [ ]:
answer = run_react_agent(
    "Search the web for information about AI agents and their main characteristics."
)

print("\nFinal Answer:")
print(answer)

# Multi-Step Research

## Multi-Step Research

A ReAct agent can perform multiple tool calls for a single task.

For example, instead of searching for only one topic, we can ask the agent to research two related topics.

The agent can:

1. Search for the first topic
2. Observe the results
3. Decide what information is needed next
4. Search again
5. Observe the new results
6. Stop when the allowed steps are completed

In [ ]:
answer = run_react_agent(
    "Research LangChain and LangGraph. Find information about what they are and how they are related."
)

print("\nFinal Answer:")
print(answer)

# Understand the Agent Loop

## Understanding the Agent Loop

Our agent follows the ReAct pattern:

Reason → Act → Observe → Repeat

### Reason

The LLM decides what needs to be done.

### Act

The agent selects and executes a tool.

### Observe

The agent receives the tool result.

### Repeat

The agent uses the new information to decide what to do next.

### Terminate

The loop stops when the agent provides an answer or reaches the maximum number of allowed steps.

# Why Do We Need MAX_STEPS?

## Termination Conditions

An agent needs a termination condition.

Without one, an agent could continue calling tools repeatedly.

In our implementation, we use:

MAX_STEPS = 2

This means the agent can perform up to two reasoning and tool-use cycles.

This gives us a simple safety limit for our activity.

# Key Takeaways

In this activity, you built a basic ReAct agent.

You learned that an agent can:

- Reason about a task
- Select an appropriate tool
- Execute the tool
- Observe the result
- Use the result to decide what to do next
- Repeat the process
- Stop when a termination condition is reached

The core ReAct pattern is:

Reason → Act → Observe → Repeat